# SAM 3 resistor dataset refinement + merge

Downloads the three COCO resistor datasets directly from the `resistor_model` GitHub `r1` release, refines their existing annotations with SAM 3, removes images with no accepted SAM 3 annotation, deduplicates exact duplicate images, and merges/splits the result 80/10/10.

Use a GPU runtime. The workflow asks only for Hugging Face authentication; the datasets are downloaded automatically. The main cell prints stage headers, download progress, SAM 3 image/instance counts, throughput/ETA, and export progress.

After the dataset workflow finishes, run the final test cell to upload any image and inspect SAM 3's resistor segmentation result.

In [ ]:
import urllib.request
print('[launcher] downloading latest SAM 3 merge workflow...', flush=True)
url = 'https://raw.githubusercontent.com/Persie0/resistor_model/main/colab/sam3_merge_3_resistor_datasets.py'
path = '/content/sam3_merge_3_resistor_datasets.py'
urllib.request.urlretrieve(url, path)
print('[launcher] starting workflow...', flush=True)
exec(compile(open(path, encoding='utf-8').read(), path, 'exec'))


## Test SAM 3 on one uploaded image

Upload one resistor image. This uses the already loaded SAM 3 model with the text prompt `resistor body` and visualizes every returned mask with its score and bounding box.

In [ ]:
import io
import time
import matplotlib.pyplot as plt
from google.colab import files

print('[test 1/4] Upload one image...', flush=True)
uploaded_test = files.upload()
if not uploaded_test:
    raise RuntimeError('No image uploaded.')

test_name, test_bytes = next(iter(uploaded_test.items()))
print(f'[test 2/4] Loading {test_name} ({len(test_bytes) / 1024:.1f} KiB)...', flush=True)
test_image = Image.open(io.BytesIO(test_bytes)).convert('RGB')
print(f'[test] image size: {test_image.width}x{test_image.height}', flush=True)

print('[test 3/4] Running SAM 3 with prompt: resistor body', flush=True)
test_started = time.monotonic()
test_state = processor.set_image(test_image)
test_out = processor.set_text_prompt(prompt='resistor body', state=test_state)
test_elapsed = time.monotonic() - test_started

test_masks = test_out['masks'].squeeze(1).detach().cpu().numpy().astype(np.uint8)
test_boxes = test_out['boxes'].detach().cpu().numpy()
test_scores = test_out['scores'].detach().cpu().numpy().astype(float)

print(f'[test] inference complete in {test_elapsed:.2f}s | detections: {len(test_masks)}', flush=True)
for i, (box, score) in enumerate(zip(test_boxes, test_scores)):
    print(f'[test] detection {i}: score={score:.4f}, box={box.tolist()}', flush=True)

print('[test 4/4] Rendering result...', flush=True)
base = np.array(test_image).copy()
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(base)

for i, (mask, box, score) in enumerate(zip(test_masks, test_boxes, test_scores)):
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[..., 3] = mask.astype(np.float32) * 0.35
    overlay[..., 0] = mask.astype(np.float32)
    ax.imshow(overlay)

    x1, y1, x2, y2 = map(float, box)
    rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(x1, max(0, y1 - 5), f'{i}: {score:.3f}', fontsize=10, backgroundcolor='white')

ax.set_title(f'SAM 3 resistor-body detections: {test_name}')
ax.axis('off')
plt.show()
print('[test] done', flush=True)
